# Experiment 11c — OOF-Optimised Blend: GRU + LSTM + exp-7c

This notebook loads the saved OOF predictions from:
- `oof_predictions_11a_gru.csv` (GRU encoder-decoder, exp-11a)
- `oof_predictions_11b_lstm.csv` (LSTM encoder-decoder, exp-11b)
- `oof_predictions_7b_blend.csv` (best prior blend: 0.285 LGBM + 0.715 MLP, exp-7c / exp-7b)

and finds OOF-optimal blend weights via `scipy.optimize.minimize`, then assembles
the validation submission using the corresponding validation CSVs.

**Decision rule**: submit the blend only if it improves over the current best
(`val_components_7c.csv`, LB WAPE 14.68). Accept an arm if residual r < 0.90.

No training in this notebook — CPU-only, fast.

---
## §0. Paths

In [ ]:
from pathlib import Path

KAGGLE_BASE = Path("/kaggle/input/datasets/xitij27/baseline-code")
OUTPUT_DIR  = Path("/kaggle/working")
# Local override:
# KAGGLE_BASE = Path("..")
# OUTPUT_DIR  = Path(".")

# OOF predictions from 11a and 11b (written to /kaggle/working in those notebooks)
OOF_GRU_PATH  = OUTPUT_DIR / "oof_predictions_11a_gru.csv"
OOF_LSTM_PATH = OUTPUT_DIR / "oof_predictions_11b_lstm.csv"
# OOF from exp-7b blend (the base ensemble)
OOF_7C_PATH   = KAGGLE_BASE / "experiment-7" / "oof_predictions_7b_blend.csv"
# Validation predictions from 11a and 11b
VAL_GRU_PATH  = OUTPUT_DIR / "validation_predictions_11a_gru.csv"
VAL_LSTM_PATH = OUTPUT_DIR / "validation_predictions_11b_lstm.csv"
# exp-7c validation predictions (LGBM+MLP blend used for LB 14.68 submission)
VAL_7C_PATH   = KAGGLE_BASE / "experiment-7" / "val_components_7c.csv"

SUBMIT_OUT = OUTPUT_DIR / "validation_predictions_11c_blend.csv"

N_FOLDS      = 3
EVAL_HORIZON = 336

def wape(y_true, y_pred, eps=1e-8):
    import numpy as np
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

---
## §1. Load OOF Predictions

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Check all files exist before loading
for p, label in [(OOF_GRU_PATH, "11a GRU OOF"), (OOF_LSTM_PATH, "11b LSTM OOF"),
                 (OOF_7C_PATH, "exp-7c OOF")]:
    exists = p.exists()
    print(f"  {'✓' if exists else '✗'}  {label}: {p}")
    if not exists:
        raise FileNotFoundError(f"Missing: {p}. Run the corresponding notebook first.")

oof_gru  = pd.read_csv(OOF_GRU_PATH,  parse_dates=["timestamp"])
oof_lstm = pd.read_csv(OOF_LSTM_PATH, parse_dates=["timestamp"])
oof_7c   = pd.read_csv(OOF_7C_PATH,   parse_dates=["timestamp"])

print(f"\nOOF rows: GRU={len(oof_gru):,}, LSTM={len(oof_lstm):,}, 7c={len(oof_7c):,}")

# Detect prediction column names
GRU_COL  = "pred_gru"  if "pred_gru"  in oof_gru.columns  else [c for c in oof_gru.columns  if c.startswith("pred")][0]
LSTM_COL = "pred_lstm" if "pred_lstm" in oof_lstm.columns else [c for c in oof_lstm.columns if c.startswith("pred")][0]
C7_COL   = "pred_blend" if "pred_blend" in oof_7c.columns else [c for c in oof_7c.columns  if c.startswith("pred")][0]
print(f"Pred columns: GRU='{GRU_COL}', LSTM='{LSTM_COL}', 7c='{C7_COL}'")

---
## §2. Merge and Compute Pairwise Residual Correlations

In [ ]:
# Merge all three OOFs on (series_id, timestamp, fold)
merge_keys = ["series_id", "timestamp", "fold"]
joint = oof_gru[["series_id", "timestamp", "target", GRU_COL, "fold"]].copy()
joint = joint.merge(
    oof_lstm[["series_id", "timestamp", LSTM_COL, "fold"]], on=merge_keys, how="inner"
)
joint = joint.merge(
    oof_7c[["series_id", "timestamp", C7_COL, "fold"]], on=merge_keys, how="inner"
)
assert len(joint) == len(oof_gru), f"OOF merge lost rows ({len(joint)} vs {len(oof_gru)})"
print(f"Merged OOF rows: {len(joint):,}")

# Compute solo WAPEs
print("\nSolo WAPEs (OOF):")
print(f"  GRU  (11a): {wape(joint['target'], joint[GRU_COL]):.3f}")
print(f"  LSTM (11b): {wape(joint['target'], joint[LSTM_COL]):.3f}")
print(f"  7c  blend: {wape(joint['target'], joint[C7_COL]):.3f}")

# Residual correlation matrix
r_gru_lstm = np.corrcoef(joint["target"]-joint[GRU_COL], joint["target"]-joint[LSTM_COL])[0,1]
r_gru_7c   = np.corrcoef(joint["target"]-joint[GRU_COL], joint["target"]-joint[C7_COL])[0,1]
r_lstm_7c  = np.corrcoef(joint["target"]-joint[LSTM_COL], joint["target"]-joint[C7_COL])[0,1]

print("\nPairwise residual r:")
print(f"  r(GRU,  LSTM) = {r_gru_lstm:.4f}")
print(f"  r(GRU,  7c)   = {r_gru_7c:.4f}")
print(f"  r(LSTM, 7c)   = {r_lstm_7c:.4f}")

---
## §3. OOF-Optimised Blend Weights

Minimise WAPE on pooled OOF over simplex weights `(w_gru, w_lstm, w_7c)` with
`w_gru + w_lstm + w_7c = 1, all ≥ 0`.

Also tests 2-way variants (GRU+7c, LSTM+7c) in case one RNN arm hurts the blend.

In [ ]:
tgt = joint["target"].to_numpy()
gru_pred  = joint[GRU_COL].to_numpy()
lstm_pred = joint[LSTM_COL].to_numpy()
c7_pred   = joint[C7_COL].to_numpy()

def blend_wape(w, preds_list):
    w = np.array(w)
    blend = sum(w[i] * preds_list[i] for i in range(len(w)))
    return wape(tgt, blend)

def optimize_blend(preds_list, n_arms):
    """Minimise WAPE over the unit simplex."""
    best_w, best_wape_val = None, float("inf")
    n_trials = 200
    for _ in range(n_trials):
        w0 = np.random.dirichlet(np.ones(n_arms))
        res = minimize(
            lambda w: blend_wape(np.append(w, 1 - w.sum()), preds_list),
            w0[:-1],
            method="SLSQP",
            bounds=[(0.0, 1.0)] * (n_arms - 1),
            constraints={"type": "ineq", "fun": lambda w: 1.0 - w.sum()},
            options={"ftol": 1e-9, "maxiter": 500},
        )
        if res.fun < best_wape_val:
            best_wape_val = res.fun
            w_sol = np.append(res.x, 1 - res.x.sum())
            best_w = np.clip(w_sol, 0, 1)
            best_w = best_w / best_w.sum()
    return best_w, best_wape_val

np.random.seed(42)

# 3-way blend: GRU + LSTM + 7c
w3, wape3 = optimize_blend([gru_pred, lstm_pred, c7_pred], 3)
print(f"3-way (GRU + LSTM + 7c):")
print(f"  weights: GRU={w3[0]:.3f}, LSTM={w3[1]:.3f}, 7c={w3[2]:.3f}")
print(f"  WAPE: {wape3:.3f}")

# 2-way: GRU + 7c
w2g, wape2g = optimize_blend([gru_pred, c7_pred], 2)
print(f"\n2-way (GRU + 7c):")
print(f"  weights: GRU={w2g[0]:.3f}, 7c={w2g[1]:.3f}")
print(f"  WAPE: {wape2g:.3f}")

# 2-way: LSTM + 7c
w2l, wape2l = optimize_blend([lstm_pred, c7_pred], 2)
print(f"\n2-way (LSTM + 7c):")
print(f"  weights: LSTM={w2l[0]:.3f}, 7c={w2l[1]:.3f}")
print(f"  WAPE: {wape2l:.3f}")

# Best RNN only
best_rnn_wape = min(wape(tgt, gru_pred), wape(tgt, lstm_pred))
best_rnn      = "GRU" if wape(tgt, gru_pred) < wape(tgt, lstm_pred) else "LSTM"

print(f"\nSummary:")
print(f"  7c solo:        {wape(tgt, c7_pred):.3f}")
print(f"  Best RNN solo:  {best_rnn_wape:.3f} ({best_rnn})")
print(f"  2-way GRU+7c:   {wape2g:.3f}")
print(f"  2-way LSTM+7c:  {wape2l:.3f}")
print(f"  3-way blend:    {wape3:.3f}")

---
## §4. Select Best Blend & Build Submission

In [ ]:
# Pick the blend with lowest OOF WAPE
candidate_blends = [
    ("3-way GRU+LSTM+7c", w3, [gru_pred, lstm_pred, c7_pred], wape3),
    ("2-way GRU+7c",      np.array([w2g[0], w2g[1]]), [gru_pred, c7_pred], wape2g),
    ("2-way LSTM+7c",     np.array([w2l[0], w2l[1]]), [lstm_pred, c7_pred], wape2l),
    ("7c solo",           np.array([1.0]), [c7_pred], wape(tgt, c7_pred)),
]
candidate_blends.sort(key=lambda x: x[3])

best_name, best_w, best_preds, best_oof_wape = candidate_blends[0]
print(f"Selected blend: '{best_name}', OOF WAPE = {best_oof_wape:.3f}")
print(f"Weights: {best_w}")

BASELINE_OOF = wape(tgt, c7_pred)
if best_oof_wape < BASELINE_OOF:
    print(f"\n✅ Improvement: {BASELINE_OOF:.3f} → {best_oof_wape:.3f} ({BASELINE_OOF - best_oof_wape:+.3f} WAPE)")
    print("Submitting blend.")
else:
    print(f"\n⚠️  Blend ({best_oof_wape:.3f}) does not improve over 7c ({BASELINE_OOF:.3f}).")
    print("Submitting anyway — leaderboard evaluation will be the definitive test.")

---
## §5. Load Validation Predictions and Build Submission CSV

In [ ]:
# Load validation predictions for each arm
val_frames = {}
for name, path, col in [
    ("gru",  VAL_GRU_PATH,  "prediction"),
    ("lstm", VAL_LSTM_PATH, "prediction"),
    ("7c",   VAL_7C_PATH,   None),
]:
    if path.exists():
        df = pd.read_csv(path, parse_dates=["timestamp"])
        if col is None:
            # Find the prediction column
            col = [c for c in df.columns if c not in ("series_id", "timestamp")][0]
        df = df.rename(columns={col: f"pred_{name}"})
        val_frames[name] = df
        print(f"Loaded {name}: {len(df):,} rows, pred_col='{col}'")
    else:
        print(f"⚠️  {name} validation file not found: {path}")

# Merge all validation arms
merge_cols = ["series_id", "timestamp"]
val_merged = val_frames["7c"].copy()
for arm in ["gru", "lstm"]:
    if arm in val_frames:
        val_merged = val_merged.merge(
            val_frames[arm][["series_id", "timestamp", f"pred_{arm}"]],
            on=merge_cols, how="left"
        )
print(f"Merged validation rows: {len(val_merged):,}")
print(f"Columns: {list(val_merged.columns)}")

In [ ]:
# Apply selected blend weights to validation predictions
name_to_col = {"3-way GRU+LSTM+7c": ["pred_gru", "pred_lstm", "pred_7c"],
               "2-way GRU+7c":      ["pred_gru", "pred_7c"],
               "2-way LSTM+7c":     ["pred_lstm", "pred_7c"],
               "7c solo":            ["pred_7c"]}

blend_cols = name_to_col[best_name]
for col in blend_cols:
    if col not in val_merged.columns:
        raise ValueError(f"Column '{col}' not in val_merged. Make sure all arm notebooks were run.")

val_merged["prediction"] = sum(
    best_w[i] * val_merged[blend_cols[i]] for i in range(len(blend_cols))
)

submission = val_merged[["series_id", "timestamp", "prediction"]].copy()
submission.to_csv(SUBMIT_OUT, index=False)
print(f"Submission saved to {SUBMIT_OUT}")
print(f"Shape: {submission.shape}")
print(f"Sample:")
print(submission.head(5))

---
## §6. Summary

In [ ]:
print("=" * 55)
print("Experiment 11c — Blend Summary")
print("=" * 55)
print(f"Selected blend : {best_name}")
print(f"Blend weights  : {dict(zip(blend_cols, best_w.round(3)))}")
print(f"OOF WAPE       : {best_oof_wape:.3f}")
print(f"7c baseline    : {BASELINE_OOF:.3f}")
print(f"Delta          : {BASELINE_OOF - best_oof_wape:+.3f}")
print()
print("Pairwise residual correlations:")
print(f"  r(GRU,  LSTM) = {r_gru_lstm:.4f}")
print(f"  r(GRU,  7c)   = {r_gru_7c:.4f}")
print(f"  r(LSTM, 7c)   = {r_lstm_7c:.4f}")